# PDF ingestion for RAG:
- PyMuPDF (fitz) for fast extraction
- 1 chunk per page
- Page citations via metadata (page number)
- Parses filenames like:
    mama_es_novartis_guia-pacientes-CM_2025.pdf
  => topic=mama, lang=es, source=novartis, slug=guia-pacientes-CM, version=2025
- Extracts metadata from the filename
- Extracts text page by page with PyMuPDF
- Clean/normalize the extracted text a bit
- Creates one chunk per page with a stable chunk_id
- Saves everything in the docs_index.jsonl which is easy to embed, store in a vector DB, filter by metadata and cite the original page

Folder layout:
  docs/general/*.pdf
  docs/mama/*.pdf

Output:
  docs_index.jsonl   (one JSON record per page chunk)

In [4]:
import re
import json
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator, Dict, Any, List
import numpy as np

import pymupdf  # PyMuPDF

### Config

In [2]:
DOCS_ROOT = Path("docs")
TOPIC_FOLDERS = {"general", "mama"}  # extend later: {"general","mama","prostata",...}
PAGES_JSONL = Path("docs/pages.jsonl")
CHUNKS_JSONL = Path("docs/chunks.jsonl")
#Document metadata structure
@dataclass
class DocMeta:
    doc_id: str #stable unique ID
    topic: str #mama, general, prostata, etc.
    lang: str #language
    source: str #novartis, gepac, etc.
    slug: str #descriptive text
    version: str #version (v1, v2) or year of publication
    path: str #where the file is on disk
    file_hash: str #unique hash to detect changes


### Hash helpers

In [5]:
#Creates a stable fingerprint for a chunk's text
def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8", errors="ignore")).hexdigest()

# Reads the file in chunks and hashes it
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

### Text and filename normalization

In [6]:
def normalize_pdf_text(text: str) -> str:
    """Small, high-impact cleanup for PDF text."""
    if not text:
        return ""

    # Join hyphenation across line breaks: "informa-\nción" -> "información"
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Merge single newlines into spaces; keep paragraph breaks
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Collapse extra whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [7]:
def _normalize_stem(stem: str) -> str:
    # Convert common unicode dashes to ASCII hyphen
    stem = stem.replace("–", "-").replace("—", "-").replace("−", "-")
    # Normalize whitespace (just in case)
    stem = stem.strip()
    # Collapse multiple underscores
    stem = re.sub(r"_+", "_", stem)
    return stem

### Extract metadata

In [8]:
#Parses the filename using the name convention decided
def parse_filename_meta(pdf_path: Path) -> DocMeta:
    """
    Expected filename (before .pdf):
      <topic>_<lang>_<source>_<slug>_<version>

    Example:
      mama_es_novartis_guia-pacientes-CM_2025.pdf
    """
    stem_raw = pdf_path.stem
    stem = _normalize_stem(stem_raw)

    # If your slug sometimes includes underscores, this regex still works because
    # it captures "slug" as everything up to the last "_<version>"
    m = re.match(
        r"^(?P<topic>[a-z0-9-]+)_(?P<lang>[a-z]{2})_(?P<source>[a-z0-9-]+)_(?P<slug>.+)_(?P<version>[a-z0-9.-]+)$",
        stem,
        flags=re.IGNORECASE,
    )
    if not m:
        raise ValueError(
            f"Filename does not match <topic>_<lang>_<source>_<slug>_<version>.pdf : {pdf_path.name}"
        )

    topic = m.group("topic").lower()
    lang = m.group("lang").lower()
    source = m.group("source").lower()
    slug = m.group("slug")  # keep original case if you want; here we keep as-is
    version = m.group("version").lower()

    doc_id = stem.lower()  # stable and unique

    return DocMeta(
        doc_id=doc_id,
        topic=topic,
        lang=lang,
        source=source,
        slug=slug,
        version=version,
        path=str(pdf_path.as_posix()),
        file_hash=sha256_file(pdf_path),
    )

In [9]:
# Change this path to one file that exists in your project
test_path = DOCS_ROOT / "mama" / "mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf"

print("Exists:", test_path.exists())
if test_path.exists():
    meta = parse_filename_meta(test_path)
    print(meta)


Exists: True
DocMeta(doc_id='mama_en_seom-geicam-solti_clinical-guidelines_2022', topic='mama', lang='en', source='seom-geicam-solti', slug='clinical-guidelines', version='2022', path='docs/mama/mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf', file_hash='84d0bd084748ddab9e54ff373da8d846bad1ffd59b6675802bb27c0c0cbd4886')


### Find PDFs

In [10]:
#Find all pdfs
def iter_pdf_paths(root: Path) -> Iterator[Path]:
    for topic in TOPIC_FOLDERS:
        folder = root / topic
        if folder.exists():
            yield from folder.rglob("*.pdf")

In [11]:
pdfs = list(iter_pdf_paths(DOCS_ROOT))
if not pdfs:
        raise FileNotFoundError(f"No PDFs found under {DOCS_ROOT}/{{general,mama}}")
print(pdfs)

[WindowsPath('docs/mama/mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf'), WindowsPath('docs/mama/mama_es_esmo_guia-para-pacientes_v1.pdf'), WindowsPath('docs/mama/mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf'), WindowsPath('docs/mama/mama_es_novartis_guia-pacientes-CM_2025.pdf'), WindowsPath('docs/general/general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf'), WindowsPath('docs/general/general_es_pfizer_manual-pacientes_2007.pdf')]


In [12]:
p0 = pdfs[0]
print("Using:", p0)

doc = pymupdf.open(p0)
print("Pages:", doc.page_count)

page0 = doc.load_page(0)
raw = page0.get_text("text")
print("Raw sample:\n", raw[:800])

clean = normalize_pdf_text(raw)
print("\nClean sample:\n", clean[:800])
doc.close()


Using: docs\mama\mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf
Pages: 14
Raw sample:
 Vol.:(0123456789)
1 3
Clinical and Translational Oncology (2023) 25:2665–2678 
https://doi.org/10.1007/s12094-023-03203-8
CLINICAL GUIDES IN ONCOLOGY
SEOM–GEICAM–SOLTI clinical guidelines in advanced breast cancer 
(2022)
Jose Angel Garcia‑Saenz1   · Isabel Blancas2   · Isabel Echavarria3   · Carmen Hinojo4   · Mireia Margeli5 · 
Fernando Moreno1   · Sonia Pernas6   · Teresa Ramon y Cajal7   · Nuria Ribelles8   · Meritxell Bellet9
Received: 14 April 2023 / Accepted: 16 April 2023 / Published online: 6 May 2023 
© The Author(s) 2023, corrected publication 2023
Abstract
Advanced breast cancer represents a challenge for patients and for physicians due its dynamic genomic changes yielding to a 
resistance to treatments. The main goal is to improve quality of live and survival of the patients t

Clean sample:
 Vol.:(0123456789) 1 3 Clinical and Translational Oncology (2023) 25:2665–2678 https://do

### Extract page-based chunks

In [13]:
#Extracts page-based chunks
def extract_page_chunks(pdf_path: Path) -> List[Dict[str, Any]]:
    meta = parse_filename_meta(pdf_path)
    records: List[Dict[str, Any]] = []

    with pymupdf.open(pdf_path) as doc:
        for i in range(doc.page_count):
            page = doc.load_page(i)
            raw_text = page.get_text("text") #Extracts the text in a fast basic format
            text = normalize_pdf_text(raw_text) #Cleans the pdf

            #Skip empty pages
            if len(text) < 20:
                continue

            page_num = i + 1  # human-friendly, 1-based for citation
            page_id = f"{meta.doc_id}__p{page_num:03d}" #build the chunk_id

            records.append(
                {
                    "page_id": page_id,
                    "text": text,
                    "page": page_num,  # <-- use this for citations

                    # Minimal doc metadata (useful for filtering + citations)
                    "doc_id": meta.doc_id,
                    "topic": meta.topic,
                    "lang": meta.lang,
                    "source": meta.source,
                    "slug": meta.slug,
                    "version": meta.version,
                    "path": meta.path,

                    # Debug/dedupe helpers
                    "file_hash": meta.file_hash,
                    "text_hash": sha256_text(text),
                    "char_len": len(text),
                }
            )

    return records

In [14]:
# Test on one PDF
records0 = extract_page_chunks(p0)
len(records0), records0[0]["page_id"] if records0 else "No chunks (maybe filtered by ONLY_LANG)"

(14, 'mama_en_seom-geicam-solti_clinical-guidelines_2022__p001')

In [15]:
if records0:
    r = records0[0]
    print("page_id:", r["page_id"])
    print("page:", r["page"])
    print("source:", r["source"])
    print("topic:", r["topic"])
    print("\nTEXT SAMPLE:\n", r["text"][:1200])


page_id: mama_en_seom-geicam-solti_clinical-guidelines_2022__p001
page: 1
source: seom-geicam-solti
topic: mama

TEXT SAMPLE:
 Vol.:(0123456789) 1 3 Clinical and Translational Oncology (2023) 25:2665–2678 https://doi.org/10.1007/s12094-023-03203-8 CLINICAL GUIDES IN ONCOLOGY SEOM–GEICAM–SOLTI clinical guidelines in advanced breast cancer (2022) Jose Angel Garcia‑Saenz1   · Isabel Blancas2   · Isabel Echavarria3   · Carmen Hinojo4   · Mireia Margeli5 · Fernando Moreno1   · Sonia Pernas6   · Teresa Ramon y Cajal7   · Nuria Ribelles8   · Meritxell Bellet9 Received: 14 April 2023 / Accepted: 16 April 2023 / Published online: 6 May 2023 © The Author(s) 2023, corrected publication 2023 Abstract Advanced breast cancer represents a challenge for patients and for physicians due its dynamic genomic changes yielding to a resistance to treatments. The main goal is to improve quality of live and survival of the patients through the most appropriate subsequent therapies based on the knowledge of the

In [16]:
all_pages: List[Dict[str, Any]] = []
for p in pdfs:
    try:
        pages = extract_page_chunks(p)
        all_pages.extend(pages)
        print(f"[OK] {p.name} -> {len(pages)} pages")
    except Exception as e:
        print(f"[ERR] {p.name}: {e}")

print("Total pages:", len(all_pages))

[OK] mama_en_SEOM-GEICAM-SOLTI_clinical-guidelines_2022.pdf -> 14 pages
[OK] mama_es_esmo_guia-para-pacientes_v1.pdf -> 76 pages
[OK] mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf -> 118 pages
[OK] mama_es_novartis_guia-pacientes-CM_2025.pdf -> 126 pages
[OK] general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf -> 58 pages
[OK] general_es_pfizer_manual-pacientes_2007.pdf -> 183 pages
Total pages: 575


### Extract smaller "embedding chunks"

In [17]:
def split_into_paragraphs(text: str) -> List[str]:
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if len(paras) >= 2:
        return paras
    # fallback sentence-ish
    sents = re.split(r"(?<=[\.\?\!])\s+", text.strip())
    return [s.strip() for s in sents if s.strip()]

import re
from typing import List

def pack_with_overlap(
    units: List[str],
    chunk_chars: int = 1600,
    overlap_units: int = 1,
    min_chars: int = 200,
) -> List[str]:
    if chunk_chars <= 0:
        raise ValueError("chunk_chars must be > 0")

    chunks: List[str] = []
    cur: List[str] = []

    def cur_len() -> int:
        # +2 for the "\n\n" joins between units
        return sum(len(u) for u in cur) + max(0, len(cur) - 1) * 2

    def flush_cur():
        nonlocal cur
        chunk = "\n\n".join(cur).strip()
        if len(chunk) >= min_chars:
            chunks.append(chunk)

    i = 0
    while i < len(units):
        u = units[i].strip()
        if not u:
            i += 1
            continue

        # Case 1: unit itself is bigger than chunk size → split the unit
        if len(u) > chunk_chars:
            # flush whatever we had collected so far
            if cur:
                flush_cur()
                cur = cur[-overlap_units:] if overlap_units > 0 else []

            # split this large unit into pieces (no overlap here; you can add if you want)
            for start in range(0, len(u), chunk_chars):
                piece = u[start:start + chunk_chars].strip()
                if len(piece) >= min_chars:
                    chunks.append(piece)

            i += 1
            continue

        # Case 2: cur empty → start new chunk with u
        if not cur:
            cur.append(u)
            i += 1
            continue

        # Case 3: u fits in current chunk → add it
        if cur_len() + 2 + len(u) <= chunk_chars:
            cur.append(u)
            i += 1
            continue

        # Case 4: u doesn't fit → flush current chunk
        flush_cur()

        # keep overlap, but IMPORTANT: ensure we can make progress
        cur = cur[-overlap_units:] if overlap_units > 0 else []

        # If overlap kept a chunk that still prevents u from fitting,
        # drop overlap to avoid infinite loop.
        if cur and (cur_len() + 2 + len(u) > chunk_chars):
            cur = []

        # Now loop continues without incrementing i,
        # but after flushing + possibly clearing cur, u will fit next iteration.

    # flush remainder
    if cur:
        flush_cur()

    return chunks


Modo A: Sliding Window
- Cada embedding chunk mide hasta 1600 caracteres
- El siguiente comparte unos 200 caracteres con el anterior (overlap)

``chunks = len(texto_pagina)/(chunk_chars)-overlap_chars``
- Ej: Si una página tiene 5600 caracteres => 5600/1400 = 4 chunks

Modo B: Paragraph pack (por párrafos)
1. Separar el texto en unidades 
    - ``units = split_into_paragraphs(text)``
2. Vas empaquetando párrafos en un chunk hasta llegar a chunk_chars
3. Con ``chunk_charse = 1600, overlap_units = 1``
4. Un embedding chunk tiene varios párrafos, siempre que quepan bajo max. 1600 caracteres
5. Al pasar al siguiente chunk, se repite el último párrafo del chunk anterior. 

Normalmente los modelos están cómodos con 256-5120 tokens por chunk. 1600 caracteres son aprox. 300-500 tokens

Cada embedding chunk tiene: 
- ``text`` → lo que vas a embedir
- ``chunk_id`` → identificador único
- ``page`` → número de página (cita)
- ``doc_id``, ``source``, ``topic``, ``lang``, ``path``, etc.

Esto permite: 
1. Recuperar por similityd (vector search)
2. Responder citando: source + doc_id + page

In [18]:
# Choose splitter mode: "sliding" or "paragraph"
SPLITTER_MODE = "paragraph"  # or "sliding"

SPLITTER_CFG = {
    "chunk_chars": 1600,
    "overlap_chars": 200,   # used by sliding
    "min_chars": 200,
}

PACK_CFG = {
    "chunk_chars": 1600,
    "overlap_units": 1,     # used by paragraph packing
    "min_chars": 200,
}

def page_to_embedding_chunks(page_rec: Dict[str, Any]) -> List[Dict[str, Any]]:
    #Coger el texto de la página
    text = page_rec["text"]

    #Se divide usando el splitter elegido
    #Paragraph splitter = divide por párrafos y luego empaqueta
    #Sliding window = divide con ventana deslizante por caracteres
    if SPLITTER_MODE == "paragraph":
        units = split_into_paragraphs(text)
        subs = pack_with_overlap(units, **PACK_CFG)
    else:
        subs = split_text_sliding_window(text, **SPLITTER_CFG)

    #Por cada subtexto generado (subs), crear un embedding chunk

    out: List[Dict[str, Any]] = []
    for j, sub in enumerate(subs):
        chunk_id = f'{page_rec["page_id"]}__c{j:03d}'

        out.append({
            "chunk_id": chunk_id,
            "text": sub,
            "chunk_index": j,

            # Keep page citation info
            "page": page_rec["page"],
            "page_id": page_rec["page_id"],

            # Copy doc metadata
            "doc_id": page_rec["doc_id"],
            "topic": page_rec["topic"],
            "lang": page_rec["lang"],
            "source": page_rec["source"],
            "slug": page_rec["slug"],
            "version": page_rec["version"],
            "path": page_rec["path"],

            # Debug helpers
            "file_hash": page_rec["file_hash"],
            "text_hash": sha256_text(sub),
            "char_len": len(sub),
        })

    return out


## Write JSONL output

In [19]:
def write_jsonl(records: List[Dict[str, Any]], out_path: Path) -> None:
    with out_path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [20]:
#Build embedding chunks from all pages
all_chunks: List[Dict[str, Any]] = []
for pg in all_pages:
    all_chunks.extend(page_to_embedding_chunks(pg))

print("Total embedding chunks:", len(all_chunks))
write_jsonl(all_chunks, CHUNKS_JSONL)
print("Wrote:", CHUNKS_JSONL.resolve())

Total embedding chunks: 909
Wrote: C:\Users\mamen\Documents\Python\RAGChatBot_LOCAL\docs\chunks.jsonl


In [21]:
write_jsonl(all_pages, PAGES_JSONL)
print("Wrote:", PAGES_JSONL.resolve())

Wrote: C:\Users\mamen\Documents\Python\RAGChatBot_LOCAL\docs\pages.jsonl


In [22]:
lengths = [c["char_len"] for c in all_chunks]
print("chunks:", len(lengths))
print("min:", min(lengths), "avg:", sum(lengths)/len(lengths), "max:", max(lengths))

# cuántos chunks por página (aprox)
from collections import Counter
by_page = Counter(c["page_id"] for c in all_chunks)
print("pages:", len(by_page))
print("chunks/page: min", min(by_page.values()), "avg", sum(by_page.values())/len(by_page), "max", max(by_page.values()))


chunks: 909
min: 200 avg: 1157.6039603960396 max: 1600
pages: 551
chunks/page: min 1 avg 1.6497277676950999 max 5


In [23]:
#Inspect one chunk
if all_chunks:
    ex = all_chunks[12]
    print("chunk_id:", ex["chunk_id"])
    print("doc_id:", ex["doc_id"])
    print("page:", ex["page"])
    print("source:", ex["source"])
    print("\nTEXT SAMPLE:\n", ex["text"][:])

chunk_id: mama_en_seom-geicam-solti_clinical-guidelines_2022__p005__c002
doc_id: mama_en_seom-geicam-solti_clinical-guidelines_2022
page: 5
source: seom-geicam-solti

TEXT SAMPLE:
 Alpelisib is now approved by the EMA for patients progressing on endocrine monotherapy, although in other countries it is approved irrespectively of prior iCDK4/6 treatment [I, B] [10, 11].

Everolimus in combination with AI or fulvestrant is also an option for second-line treatment [I, B], with a PFS benefit of 6.9 vs 2.8 months with exemestane and everolimus, and 10.3 vs.

5.1 months with fulvestrant and everolimus over fulvestrant monotherapy.

However, this latter PFS benefit could be overestimated due to a high level of informative censoring [I, B] [12].

Of note, these two combinations were tested before the use of iCDK4/6, so their efficacy data after iCDK4/6 are scarce.

Chemotherapy Chemotherapy is still a mainstay in the treatment of advanced HR+/HER2- breast cancer [I, A].

No specific algorithm f

You can cite pages later like: Fuente: {source} ({doc_id}), p.{page}

### load all_chunks and all_pages

In [24]:
loaded_pages = []
with PAGES_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        loaded_pages.append(json.loads(line))

print("Loaded pages:", len(loaded_pages))
assert loaded_pages == all_pages, "Loaded pages do not match original!"
print("Complete integrity check for pages passed.")

Loaded pages: 575
Complete integrity check for pages passed.


In [ ]:
loaded_chunks = []
with CHUNKS_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        loaded_chunks.append(json.loads(line))

print("Loaded chunks:", len(loaded_chunks))
assert loaded_chunks == all_chunks, "Loaded chunks do not match original!"
print("Complete integrity check for chunks passed.")

#Loaded chunks: 909
#Complete integrity check for chunks passed.


Loaded chunks: 909


# Compute Embeddings

In [ ]:
# If needed:
#Install + load model

# !pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer #Uses pytorch, not tensorflow

EMB_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
#Embedding size = 384
#Maximum input length = 512 tokens
emb_model = SentenceTransformer(EMB_MODEL_NAME)


c:\Users\mamen\anaconda3\envs\RAGChatBot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\mamen\anaconda3\envs\RAGChatBot\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mamen\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pytho

In [27]:
#Embed all chunks
import numpy as np

texts = [c["text"] for c in all_chunks]
emb = emb_model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print("Embeddings shape:", emb.shape)


Batches: 100%|██████████| 15/15 [00:21<00:00,  1.42s/it]

Embeddings shape: (909, 384)


In [28]:
#Save metadata+text, and save vectors in .npy

# Save vectors
np.save("docs/chunks_vectors.npy", emb)

# Save metadata without vectors
META_JSONL = Path("docs/chunks_meta.jsonl")
meta_only = [{k:v for k,v in c.items() if k != "vector"} for c in all_chunks]
write_jsonl(meta_only, META_JSONL)

print("Wrote: chunks_vectors.npy and", META_JSONL.resolve())


Wrote: chunks_vectors.npy and C:\Users\mamen\Documents\Python\RAGChatBot_LOCAL\docs\chunks_meta.jsonl


# Sanity checks

In [ ]:
import numpy as np, json
from pathlib import Path

vecs = np.load("docs/chunks_vectors.npy")
meta_path = Path("docs/chunks_meta.jsonl")

# count lines in jsonl
n_meta = sum(1 for _ in meta_path.open("r", encoding="utf-8"))

print("Vectors shape:", vecs.shape)
print("Metadata rows:", n_meta)

assert vecs.shape[0] == n_meta, "Mismatch: number of vectors != number of metadata rows"
print("OK: vectors and metadata aligned by row index")


Vectors shape: (909, 384)
Metadata rows: 909
OK: vectors and metadata aligned by row index


In [32]:
#Check for NaNs, Infs, and normalization

print("Any NaN:", np.isnan(vecs).any())
print("Any Inf:", np.isinf(vecs).any())

norms = np.linalg.norm(vecs, axis=1)
print("Norms: min", norms.min(), "avg", norms.mean(), "max", norms.max())

# Expect ~1.0 average norm


Any NaN: False
Any Inf: False
Norms: min 0.9999999 avg 1.0 max 1.0000001


### Quick semantic retrieval test (no vector DB yet)

In [33]:
# Load metadata into a list (same order as vecs)
meta = []
with open("docs/chunks_meta.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        meta.append(json.loads(line))

def search(query: str, top_k: int = 5):
    q = emb_model.encode([query], normalize_embeddings=True)[0]
    scores = vecs @ q  # cosine similarity because vectors are normalized
    idx = np.argsort(-scores)[:top_k]
    for rank, i in enumerate(idx, 1):
        m = meta[i]
        print(f"\n#{rank} score={scores[i]:.4f} | {m['source']} | {m['doc_id']} | p.{m['page']} | {m['chunk_id']}")
        print(m["text"][:500])

search("efectos secundarios de la quimioterapia", top_k=5)



#1 score=0.7343 | novartis | mama_es_novartis_guia-pacientes-cm_2025 | p.89 | mama_es_novartis_guia-pacientes-cm_2025__p089__c001
Todo ello también puede afectar a la imagen corporal y al funcionamiento sexual, por lo que la radioterapia puede alterar la ca­lidad de vida relacionada con la salud incluido el bienestar sexual.

Impacto de la quimioterapia Los efectos adversos de la quimioterapia son importantes, pero se ha de tener en cuenta que son temporales y la mayoría se solucionan en el plazo de un año de finalizar el tratamiento.

Son frecuentes efectos secundarios como: • Pérdida del cabello.

• Fatiga.

• Pérdida de

#2 score=0.7326 | gepac | general_es_gepac_guia-toxicidad-quimioterapia_v1 | p.5 | general_es_gepac_guia-toxicidad-quimioterapia_v1__p005__c000
www.gepac.es 3 INTRODUCCIÓN La quimioterapia forma parte del tratamiento de la mayoría de las enfermedades oncológicas en algún momento de su evolución.

Los fármacos quimioterápicos pueden administrarse con distintos objet